# Inventario de 292 Genéricos INPC 2024
## Análisis exploratorio y conexión con el proyecto G-CPI

**Archivo fuente:** `Inventario INPC_292 genéricos.xlsx`  
**Fuente oficial:** INEGI — Actualización de Canasta y Ponderadores INPC 2024 (publicada 22 agosto 2024)  
**Ponderadores de subgenéricos:** ENIGH E_AVPR 1ª quincena noviembre 2023  

---
Corre **Kernel → Restart & Run All** para ejecutar todo de una vez.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELDA 1 — IMPORTS Y CARGA DEL ARCHIVO
# ═══════════════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
import glob
from pathlib import Path

OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

# Detectar archivo automáticamente
candidatos = glob.glob('Inventario INPC*.xlsx') + glob.glob('Inventario_INPC*.xlsx')
if not candidatos:
    raise FileNotFoundError(
        '❌ No se encontró el archivo Excel. '
        'Asegúrate de que esté en la misma carpeta que este notebook. '
        f'Archivos en carpeta: {glob.glob("*.xlsx")}'
    )
FILENAME = candidatos[0]
print(f'✅ Archivo detectado: {FILENAME}')

# Cargar
wb   = openpyxl.load_workbook(FILENAME, read_only=True)
ws   = wb.active
rows = list(ws.iter_rows(values_only=True))

# Fila índice 3 = encabezados
headers = rows[3]
data    = [r for r in rows[4:] if any(x is not None for x in r)]
df      = pd.DataFrame(data, columns=headers)
df['num'] = pd.to_numeric(df['No. de gen.'], errors='coerce')

print(f'   Filas totales    : {len(df):,}')
print(f'   Genéricos únicos : {df["No. de gen."].nunique()}')
print(f'   Columnas         : {len(df.columns)}')
print()
for c in df.columns:
    print(f'  {c}')
df.head(5)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELDA 2 — SITUACIÓN DE LOS GENÉRICOS vs. CANASTA 2018
# ═══════════════════════════════════════════════════════════════════════════════
sit = df.drop_duplicates('No. de gen.')['Situación del genérico'].value_counts()

print('═' * 52)
print('  SITUACIÓN DE LOS 292 GENÉRICOS vs. CANASTA 2018')
print('═' * 52)
for k, v in sit.items():
    pct = v / 292 * 100
    print(f'  {k:<15} {v:>4} genéricos  ({pct:.1f}%)')
print('─' * 52)
print(f'  TOTAL          {sit.sum():>4}')
print()
print('Contexto: canasta 2018 tenía 299 genéricos → 2024 tiene 292')
print('  • 259 permanecieron Igual')
print('  •  20 se Desagregaron (ej: Streaming, Bebidas energéticas, Cilantro)')
print('  •  20 se Fusionaron en 9 genéricos (ej: Camarón + Otros mariscos)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELDA 3 — MAPEO A DIVISIONES CCIF 2018
# ═══════════════════════════════════════════════════════════════════════════════
div_manual = {
    **{i: '01' for i in range(1,   100)},   # Alimentos y beb. no alcohólicas
    **{i: '02' for i in range(100, 110)},   # Bebidas alcohólicas y tabaco
    **{i: '03' for i in range(110, 138)},   # Vestido y calzado
    **{i: '04' for i in range(138, 147)},   # Vivienda
    **{i: '05' for i in range(147, 184)},   # Muebles y hogar
    **{i: '06' for i in range(184, 206)},   # Salud
    **{i: '07' for i in range(206, 228)},   # Transporte
    271: '07',                              # Seguro de automóvil → Transporte
    **{i: '08' for i in range(228, 238)},   # Comunicación
    **{i: '09' for i in range(238, 256)},   # Recreación y cultura
    **{i: '10' for i in range(256, 263)},   # Educación
    **{i: '11' for i in range(263, 271)},   # Restaurantes y hoteles
    **{i: '12' for i in [287,288,289,290,291,292]},  # Bienes y servicios diversos
    **{i: '13' for i in range(272, 287)},   # Cuidado personal
}

DIV_NOMBRES = {
    '01': 'Alimentos y beb. no alcohólicas',
    '02': 'Bebidas alcohólicas y tabaco',
    '03': 'Vestido y calzado',
    '04': 'Vivienda',
    '05': 'Muebles y hogar',
    '06': 'Salud',
    '07': 'Transporte',
    '08': 'Comunicación',
    '09': 'Recreación y cultura',
    '10': 'Educación',
    '11': 'Restaurantes y hoteles',
    '12': 'Bienes y servicios diversos',
    '13': 'Cuidado personal',
}

df['ccif_div']  = df['num'].map(div_manual)
uniq            = df.drop_duplicates('No. de gen.').copy()
sin_div         = uniq['ccif_div'].isna().sum()

por_div = (
    uniq.groupby('ccif_div')
    .agg(n_genericos=('No. de gen.', 'count'))
    .reset_index()
)
por_div['nombre_div'] = por_div['ccif_div'].map(DIV_NOMBRES)
por_div = por_div.sort_values('ccif_div')

print('═' * 62)
print('  GENÉRICOS POR DIVISIÓN CCIF')
print('═' * 62)
for _, r in por_div.iterrows():
    bar = '█' * r['n_genericos']
    print(f"  {r['ccif_div']}  {r['nombre_div']:<32} {r['n_genericos']:>3}  {bar}")
print('─' * 62)
print(f"  TOTAL{' ' * 35} {por_div['n_genericos'].sum():>3}")
print(f"  Sin asignar: {sin_div}")

por_div.to_csv(OUT_DIR / 'inv292_por_division.csv', index=False)
print('\n✅ Guardado: outputs/inv292_por_division.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELDA 4 — SUBGENÉRICOS CON PONDERACIÓN INTERNA
# ═══════════════════════════════════════════════════════════════════════════════
POND_COL = 'Ponderación Subgenéricos (ENIGH E_AVPR 1q nov 23)'
df['pond_subgen'] = pd.to_numeric(df[POND_COL], errors='coerce')

subgen = df[df['pond_subgen'].notna()].copy()
subgen = subgen[['No. de gen.', 'Nombre del Genérico',
                  'Subgenéricos', 'pond_subgen', 'ccif_div']].copy()
subgen['div_nombre'] = subgen['ccif_div'].map(DIV_NOMBRES)

print(f'Total subgenéricos ponderados : {len(subgen)}')
print(f'Genéricos que los contienen  : {subgen["No. de gen."].nunique()}')
print()

for gen_num in subgen['No. de gen.'].unique():
    sub    = subgen[subgen['No. de gen.'] == gen_num]
    nombre = sub['Nombre del Genérico'].iloc[0]
    div    = sub['ccif_div'].iloc[0]
    print(f'  [{gen_num}] {nombre}  (Div {div} — {DIV_NOMBRES.get(div,"")})')
    for _, r in sub.iterrows():
        bar = '▓' * int(r['pond_subgen'] / 5)
        print(f'       {r["Subgenéricos"]:<42} {r["pond_subgen"]:>6.2f}%  {bar}')
    print()

subgen.to_csv(OUT_DIR / 'inv292_subgenericos_ponderados.csv', index=False)
print('✅ Guardado: outputs/inv292_subgenericos_ponderados.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELDA 5 — GENÉRICOS CON PERFIL DE GÉNERO O DEMOGRÁFICO
# ═══════════════════════════════════════════════════════════════════════════════
nombres_uniq = df.drop_duplicates('No. de gen.')[
    ['No. de gen.', 'Nombre del Genérico', 'ccif_div']].copy()

kw_mujer  = ['mujer', 'femenin', 'niña', 'faldas', 'vestido', 'blusas',
              'medias', 'pantimedias', 'sanitarias', 'maquillaje',
              'sala de belleza', 'embarazo']
kw_hombre = ['hombre', 'masculin', 'camisa', 'traje', 'afeitar',
              'navajas', 'barbacoa', 'corte de cabello']
kw_nino   = ['niño', 'niña', 'bebé', 'escolar', 'guardería',
              'infantil', 'preescolar', 'primaria', 'secundaria']

def clasif(nombre):
    n = nombre.lower()
    flags = []
    if any(k in n for k in kw_mujer):  flags.append('👩 Mujer')
    if any(k in n for k in kw_hombre): flags.append('👨 Hombre')
    if any(k in n for k in kw_nino):   flags.append('👶 Niño/a')
    return ', '.join(flags) if flags else 'General'

nombres_uniq['demografico'] = nombres_uniq['Nombre del Genérico'].apply(clasif)
especificos = nombres_uniq[nombres_uniq['demografico'] != 'General'].copy()
especificos['div_nombre'] = especificos['ccif_div'].map(DIV_NOMBRES)

print('═' * 68)
print('  GENÉRICOS CON PERFIL DEMOGRÁFICO EXPLÍCITO EN SU NOMBRE')
print('═' * 68)
print(f'  Total: {len(especificos)} de 292 genéricos\n')
for _, r in especificos.sort_values('ccif_div').iterrows():
    print(f"  {r['No. de gen.']}  {r['Nombre del Genérico']:<52} {r['demografico']}")

print('\n  Resumen por categoría:')
for cat, cnt in especificos['demografico'].value_counts().items():
    print(f'    {cat}: {cnt} genéricos')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELDA 6 — GRÁFICA: GENÉRICOS POR DIVISIÓN Y SITUACIÓN
# ═══════════════════════════════════════════════════════════════════════════════
sit_div = (
    uniq.groupby(['ccif_div', 'Situación del genérico'])
    .size().reset_index(name='n')
    .pivot_table(index='ccif_div', columns='Situación del genérico',
                 values='n', fill_value=0)
    .reset_index()
)
sit_div['nombre'] = sit_div['ccif_div'].map(DIV_NOMBRES)
sit_div = sit_div.set_index('nombre')

cols_sit = [c for c in ['Igual', 'Desagregado', 'Fusionado'] if c in sit_div.columns]
colores  = {'Igual': '#4CAF50', 'Desagregado': '#2196F3', 'Fusionado': '#FF9800'}

fig, ax = plt.subplots(figsize=(12, 7))
bottom = np.zeros(len(sit_div))
for col in cols_sit:
    vals = sit_div[col].values
    ax.barh(sit_div.index, vals, left=bottom,
            color=colores[col], label=col, alpha=0.85)
    bottom += vals
for i, total in enumerate(bottom):
    ax.text(total + 0.3, i, str(int(total)), va='center', fontsize=9)

ax.set_xlabel('Número de genéricos', fontsize=11)
ax.set_title('Composición de la Canasta INPC 2024 por División CCIF\n'
             'Situación respecto a la canasta 2018',
             fontsize=12, fontweight='bold')
ax.legend(title='Situación vs. 2018', fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'inv292_genericos_por_division.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Guardado: outputs/inv292_genericos_por_division.png')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELDA 7 — GENÉRICOS EN DIVISIONES CON MAYOR DIFERENCIAL G-CPI
# ═══════════════════════════════════════════════════════════════════════════════
divs_alto_diferencial = {
    '11': ('Restaurantes y hoteles', 'H.solo 15.7%  vs  M.sola 10.3%'),
    '13': ('Cuidado personal',       'H.solo 11.1%  vs  M.sola  6.4%'),
    '01': ('Alimentos',              'M.sola 19.0%  vs  H.solo 13.7%'),
    '07': ('Transporte',             'H.c/pareja 20.9%  vs  M.sola 15.7%'),
    '10': ('Educación',              'H.c/pareja 8.9%   vs  H.solo  3.9%'),
}

kw_genero = ['mujer', 'hombre', 'niño', 'niña', 'bebé',
             'embarazo', 'afeitar', 'belleza', 'sanitaria']

print('═' * 72)
print('  GENÉRICOS EN DIVISIONES CON MAYOR DIFERENCIAL ENTRE GRUPOS G-CPI')
print('═' * 72)

for div, (nombre, nota) in divs_alto_diferencial.items():
    gen_div = uniq[uniq['ccif_div'] == div].copy()
    print(f'\n  División {div} — {nombre}  |  Diferencial: {nota}')
    print(f'  {len(gen_div)} genéricos:')
    for _, r in gen_div.iterrows():
        tag = '  ◄ PERFIL DE GÉNERO' if any(
            k in r['Nombre del Genérico'].lower() for k in kw_genero) else ''
        print(f"    {r['No. de gen.']}  {r['Nombre del Genérico']}{tag}")

# Exportar tabla completa
tabla = uniq[['No. de gen.', 'Nombre del Genérico',
              'ccif_div', 'Situación del genérico']].copy()
tabla['div_nombre'] = tabla['ccif_div'].map(DIV_NOMBRES)
tabla.to_csv(OUT_DIR / 'inv292_tabla_completa_ccif.csv', index=False)
print('\n✅ Guardado: outputs/inv292_tabla_completa_ccif.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELDA 8 — GRÁFICA: GENÉRICOS CON PERFIL DEMOGRÁFICO POR DIVISIÓN
# ═══════════════════════════════════════════════════════════════════════════════
gen_demo     = especificos.copy()
demo_por_div = gen_demo.groupby('div_nombre').size().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(demo_por_div.index, demo_por_div.values, color='#9C27B0', alpha=0.85)
for bar, val in zip(bars, demo_por_div.values):
    ax.text(val + 0.05, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=10)

ax.set_xlabel('Número de genéricos con perfil demográfico', fontsize=11)
ax.set_title(
    'Genéricos INPC 2024 con referencia de género o grupo demográfico\n'
    'La canasta oficial los agrega con un ponderador nacional único — el G-CPI no',
    fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'inv292_genericos_demograficos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Guardado: outputs/inv292_genericos_demograficos.png')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELDA 9 — RESUMEN EJECUTIVO
# ═══════════════════════════════════════════════════════════════════════════════
n_genero    = len(especificos)
n_igual     = sit.get('Igual', 0)
n_desag     = sit.get('Desagregado', 0)
n_fusion    = sit.get('Fusionado', 0)

print('═' * 72)
print('  RESUMEN EJECUTIVO — INVENTARIO 292 GENÉRICOS INPC 2024')
print('  Relevancia para el proyecto G-CPI')
print('═' * 72)
print(f"""
FUENTE OFICIAL
  INEGI — Actualización Canasta y Ponderadores INPC 2024 (22 ago 2024)
  Ponderadores derivados de: ENIGH Estacional 2022 ← misma fuente que el G-CPI ✅

CANASTA 2024
  {por_div['n_genericos'].sum()} genéricos en 13 divisiones CCIF 2018
  Vs. canasta 2018: {n_igual} iguales · {n_desag} desagregados · {n_fusion} fusionados

HALLAZGO CLAVE PARA EL G-CPI
  {n_genero} de los 292 genéricos tienen perfil demográfico explícito
  (ropa de mujer, ropa de hombre, consulta durante el embarazo, etc.)
  → El INPC oficial los pondera con un peso NACIONAL ÚNICO
  → El G-CPI revela que esos pesos difieren por tipo de jefatura

CONEXIÓN CON FEEDBACK DE ARTHUR
  Punto 1 — Comparar G-CPI vs índice oficial:
    Ambos usan ENIGH 2022 como fuente, pero el INPC tiene una canasta
    nacional única. La brecha G-CPI vs INPC cuantifica el sesgo de no
    diferenciar por jefatura.

  Punto 2 — Canasta especializada por grupo:
    Este inventario es el mapa completo de los 292 genéricos disponibles.
    Una extensión futura calcularía inflación a nivel genérico × grupo.

OUTPUTS GENERADOS EN outputs/
  inv292_por_division.csv           — genéricos por división CCIF
  inv292_subgenericos_ponderados.csv — subgenéricos con ponderación interna
  inv292_tabla_completa_ccif.csv    — los 292 genéricos con su división
  inv292_genericos_por_division.png — gráfica de composición por situación
  inv292_genericos_demograficos.png — gráfica de genéricos con perfil de género
""")
print('✅ Análisis completo.')